# PJM Stack Model — PUDL Data Validation

**Objective:** Validate the PJM Stack Model workbook's capacity inputs against
publicly available PUDL datasets (EIA 860, EIA 923, FERC Form 1, EPA CEMS).

**Workbook:** `.excel/PJM_Stack_Model_v1_2026_mar_10.xlsx`  
**PUDL source:** `s3://pudl.catalyst.coop/nightly` (Apache Parquet)  
**Reference:** `.markdowns/PUDL.md`

**Validations performed:**
1. Capacity totals by fuel type (workbook vs EIA 860)
2. Generator-level summer capacity comparison
3. Heat rate comparison (workbook vs EIA implied heat rates)
4. Fuel cost assumptions vs EIA 923 delivered fuel costs
5. VOM reasonableness check vs FERC Form 1 nonfuel O&M
6. Fuel category / prime mover classification check
7. Operational status — retired or missing units
8. Wind/Solar installed capacity totals
9. Pass/Fail summary

## 1. Setup & Data Loading

### Imports & Configuration

In [120]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

WORKBOOK = Path("../../.excel/PJM_Stack_Model_v1_2026_mar_10.xlsx")
assert WORKBOOK.exists(), f"Workbook not found: {WORKBOOK}"

PARQUET_PATH = "s3://pudl.catalyst.coop/nightly"

FUEL_COLORS = {
    "Nuclear":   "#7b2d8e",
    "Wind":      "#2ca02c",
    "Solar":     "#f5c542",
    "Hydro":     "#1f77b4",
    "Biomass":   "#8c564b",
    "Coal":      "#555555",
    "Gas CC":    "#d62728",
    "Gas CT/ST": "#ff7f0e",
    "Oil":       "#17becf",
    "Storage":   "#9467bd",
    "Other":     "#bcbd22",
}

# Validation results collector
validation_results = []

In [121]:
def read_pudl(table_name: str, columns: list[str] | None = None) -> pd.DataFrame:
    """Read a PUDL table from the nightly S3 bucket."""
    path = f"{PARQUET_PATH}/{table_name}.parquet"
    return pd.read_parquet(path, columns=columns, dtype_backend="pyarrow")


def record_result(name: str, status: str, detail: str):
    """Record a pass/fail/warn validation result."""
    validation_results.append({"check": name, "status": status, "detail": detail})
    icon = {"PASS": "\u2705", "FAIL": "\u274c", "WARN": "\u26a0\ufe0f"}.get(status, "")
    print(f"{icon} [{status}] {name}: {detail}")

### 1a. Load Workbook — Stack Model

In [122]:
stack_raw = pd.read_excel(
    WORKBOOK, sheet_name="Stack Model", header=2,
    usecols="A:Q",
)

# Drop hub-header rows (NaN in Plant Name)
stack_raw = stack_raw.dropna(subset=["Plant Name"])

col_map = {
    stack_raw.columns[0]:  "power_hub",
    stack_raw.columns[1]:  "plant_name",
    stack_raw.columns[2]:  "fuel_category",
    stack_raw.columns[3]:  "unit_type",
    stack_raw.columns[4]:  "summer_cap_mw",
    stack_raw.columns[5]:  "min_load_mw",
    stack_raw.columns[6]:  "heat_rate",
    stack_raw.columns[7]:  "fuel_hub",
    stack_raw.columns[8]:  "fuel_price",
    stack_raw.columns[9]:  "fuel_cost_orig",
    stack_raw.columns[10]: "vom",
    stack_raw.columns[11]: "carbon_cost",
    stack_raw.columns[12]: "mc_orig",
    stack_raw.columns[13]: "cum_cap_hub",
    stack_raw.columns[14]: "must_run",
    stack_raw.columns[15]: "on_off",
    stack_raw.columns[16]: "cum_cap_system",
}
stack_raw = stack_raw.rename(columns=col_map)

# Keep only "On" units
wb = stack_raw[stack_raw["on_off"] == 1].copy()

for c in ["heat_rate", "fuel_price", "vom", "carbon_cost", "summer_cap_mw", "min_load_mw"]:
    wb[c] = pd.to_numeric(wb[c], errors="coerce").fillna(0)

wb["must_run_flag"] = wb["must_run"].astype(str).str.strip().str.lower() == "yes"

print(f"Loaded {len(wb):,} active units from Stack Model")
print(f"  Fuel categories: {sorted(wb['fuel_category'].dropna().unique())}")
wb[["power_hub", "plant_name", "fuel_category", "summer_cap_mw",
    "heat_rate", "fuel_price", "vom", "must_run"]].head(10)

Loaded 4,593 active units from Stack Model
  Fuel categories: ['Biomass', 'Coal', 'Gas CC', 'Gas CT/ST', 'Hydro', 'Nuclear', 'Oil', 'Other', 'Solar', 'Storage', 'Wind']


,power_hub,plant_name,fuel_category,summer_cap_mw,heat_rate,fuel_price,vom,must_run
1,PJM AE,"PSEG Salem Generating Station (Salem, NJ)",Oil,38.407943,0.0,16.0,5.80,Yes
2,PJM AE,"Paulsboro Refinery (Gloucester, NJ)",Gas CT/ST,27.000000,0.0,2.6,4.78,Yes
3,PJM AE,"Paulsboro Refinery (Gloucester, NJ)",Gas CT/ST,14.500000,0.0,2.6,5.23,Yes
4,PJM AE,"Paulsboro Refinery (Gloucester, NJ)",Gas CT/ST,14.500000,0.0,2.6,4.78,Yes
5,PJM AE,Jersey-Atlantic Wind Farm,Wind,7.500000,0.0,0.0,2.94,Yes
6,PJM AE,"Pennsauken Landfill (Camden, NJ)",Biomass,0.900000,0.0,0.0,2.65,Yes
7,PJM AE,"Pennsauken Landfill (Camden, NJ)",Biomass,0.900000,0.0,0.0,2.65,Yes
8,PJM AE,"Pennsauken Solar (Camden, NJ)",Solar,2.400000,0.0,0.0,1.00,Yes
9,PJM AE,"Atlantic City Convention Center (Atlantic, NJ)",Solar,0.500000,0.0,0.0,1.09,Yes
10,PJM AE,"Atlantic City Convention Center (Atlantic, NJ)",Solar,0.500000,0.0,0.0,1.09,Yes


### 1b. Load Workbook — PJM Raw Data

In [123]:
raw_df = pd.read_excel(
    WORKBOOK, sheet_name="PJM Raw Data", header=1,
)
raw_df = raw_df.dropna(subset=[raw_df.columns[0]])

raw_col_map = {
    raw_df.columns[0]:  "plant_name",
    raw_df.columns[1]:  "fuel_category",
    raw_df.columns[2]:  "unit_type",
    raw_df.columns[3]:  "power_hub",
    raw_df.columns[4]:  "zone",
    raw_df.columns[5]:  "fuel_hub",
    raw_df.columns[6]:  "summer_cap_mw",
    raw_df.columns[7]:  "winter_cap_mw",
    raw_df.columns[8]:  "cap_factor",
    raw_df.columns[9]:  "heat_rate",
    raw_df.columns[10]: "vom",
    raw_df.columns[11]: "fom",
    raw_df.columns[12]: "min_load_factor",
    raw_df.columns[13]: "cold_start_hrs",
    raw_df.columns[14]: "so2_factor",
    raw_df.columns[15]: "on_off",
    raw_df.columns[16]: "baseload_mw",
    raw_df.columns[17]: "carbon_mkt",
    raw_df.columns[18]: "so2_mkt",
}
raw_df = raw_df.rename(columns=raw_col_map)

for c in ["summer_cap_mw", "winter_cap_mw", "heat_rate", "vom", "fom", "min_load_factor"]:
    raw_df[c] = pd.to_numeric(raw_df[c], errors="coerce").fillna(0)

print(f"Loaded {len(raw_df):,} units from PJM Raw Data")
raw_df.head(5)

Loaded 4,594 units from PJM Raw Data


,plant_name,fuel_category,unit_type,power_hub,zone,fuel_hub,summer_cap_mw,winter_cap_mw,cap_factor,heat_rate,vom,fom,min_load_factor,cold_start_hrs,so2_factor,on_off,baseload_mw,carbon_mkt,so2_mkt
0,"Joliet 29 (Will, IL)",Gas CT/ST,Gas OC / Steam,PJM COMED,COMED,Tetco M3,518.0,522.0,0.29,14.78,5.56,16.74,0.18,18.0,0.98,1,93.24,0,CSAPR 1
1,"Joliet 29 (Will, IL)",Gas CT/ST,Gas OC / Steam,PJM COMED,COMED,Tetco M3,518.0,522.0,0.27,14.82,5.52,16.71,0.18,18.0,1.00,0,0.00,0,CSAPR 1
2,"Christiana (New Castle, DE)",Oil,Oil,PJM DPL,DPL,NY No 2 Distillate,25.0,25.0,0.00,0.00,5.70,6.94,0.77,1.0,2.09,1,19.25,RGGI,CSAPR 1
3,"Christiana (New Castle, DE)",Oil,Oil,PJM DPL,DPL,NY No 2 Distillate,25.0,25.0,0.00,0.00,5.70,6.94,0.77,1.0,2.09,1,19.25,RGGI,CSAPR 1
4,Delaware City 10,Oil,Oil,PJM DPL,DPL,NY No 2 Distillate,18.0,18.0,0.00,16.47,5.95,8.30,0.81,1.0,1.24,1,14.58,RGGI,CSAPR 1


### 1c. Load Workbook — Assumptions

In [124]:
assumptions_raw = pd.read_excel(
    WORKBOOK, sheet_name="Assumptions", header=None,
    usecols="A:C",
)
assumptions_raw.columns = ["item", "value", "note"]
print("Assumptions sheet (first 40 rows):")
assumptions_raw.head(40)

Assumptions sheet (first 40 rows):


,item,value,note
0,PJM POWER STACK MODEL — ASSUMPTIONS,NaN,NaN
1,NaN,NaN,NaN
2,⛽ NATURAL GAS PRICES ($/MMBtu),NaN,Price ($/MMBtu)
3,Columbia TCO Pool,2.600,NaN
4,Dominion South Pt,2.450,NaN
5,Northern Ventura,2.650,NaN
6,Tetco M2 (deliveries),2.700,NaN
7,Tetco M3,2.850,NaN
8,Transco Leidy,2.550,NaN
9,Transco Z5 non-WGL,2.750,NaN


### 1d. Load PUDL — PJM Generators (EIA 860 + 923)

In [125]:
%%time        
# Identify PJM plants — core_eia860__scd_plants has iso_rto_code but NOT plant_name/state/city                                                                                                                                                                                                                                                                                                                                                        # Those fields come from out_eia__yearly_generators (loaded in the next cell)
plants_eia = read_pudl("core_eia860__scd_plants", columns=[                                                                                                                                                                                                                                                                                                                                                                                               "plant_id_eia", "iso_rto_code",
    "balancing_authority_code_eia",                                                                                                                                                                                                                                                                                                                                                                                                            
])
pjm_plant_ids = plants_eia.loc[
    (plants_eia.iso_rto_code == "PJM") | (plants_eia.balancing_authority_code_eia == "PJM"),
    "plant_id_eia"
].unique()
print(f"PJM plants in EIA 860 union filter: {len(pjm_plant_ids):,}")

PJM plants in EIA 860: 966
CPU times: total: 62.5 ms
Wall time: 651 ms


In [126]:
%%time
# Load yearly generator table and filter to PJM
gens_all = read_pudl("out_eia__yearly_generators")
gens_pjm = gens_all[gens_all.plant_id_eia.isin(pjm_plant_ids)].copy()

# Convert pyarrow date to pandas datetime for comparison
gens_pjm["report_date"] = pd.to_datetime(gens_pjm["report_date"])

# Use the most recent report year for capacity and status checks
latest_year = gens_pjm.report_date.max()
gens = gens_pjm[gens_pjm.report_date == latest_year].copy()

# Heat rate fields lag the latest EIA 860 snapshot; use the newest year with populated HR values
latest_hr_year = gens_pjm.loc[
    gens_pjm["unit_heat_rate_mmbtu_per_mwh"].notna(), "report_date"
].max()
gens_hr = gens_pjm[gens_pjm.report_date == latest_hr_year].copy()

# Filter to operating generators
gens_operating = gens[gens.operational_status == "existing"].copy()
gens_hr_operating = gens_hr[gens_hr.operational_status == "existing"].copy()

print(f"PUDL report year: {latest_year}")
print(f"Heat rate comparison year: {latest_hr_year}")
print(f"Total PJM generators (all statuses): {len(gens):,}")
print(f"Operating PJM generators: {len(gens_operating):,}")
print(f"Columns available ({len(gens.columns)}):")
print(list(gens.columns))

PUDL report year: 2026-01-01 00:00:00
Heat rate comparison year: 2024-01-01 00:00:00
Total PJM generators (all statuses): 3,188
Operating PJM generators: 2,238
Columns available (112):
['plant_id_eia', 'generator_id', 'report_date', 'unit_id_pudl', 'plant_id_pudl', 'plant_name_eia', 'utility_id_eia', 'utility_id_pudl', 'utility_name_eia', 'balancing_authority_code_eia', 'balancing_authority_name_eia', 'technology_description', 'energy_source_code_1', 'prime_mover_code', 'generator_operating_date', 'generator_retirement_date', 'operational_status', 'capacity_mw', 'fuel_type_code_pudl', 'planned_generator_retirement_date', 'capacity_factor', 'fuel_cost_per_mmbtu_source', 'fuel_cost_per_mmbtu', 'fuel_cost_per_mwh', 'unit_heat_rate_mmbtu_per_mwh', 'net_generation_mwh', 'total_fuel_cost', 'total_mmbtu', 'associated_combined_heat_power', 'bga_source', 'bypass_heat_recovery', 'carbon_capture', 'city', 'can_cofire_fuels', 'county', 'current_planned_generator_operating_date', 'data_maturity', '

---
## 2. Capacity Totals by Fuel Type

In [127]:
# -- Workbook capacity totals --
wb_cap = (
    wb.groupby("fuel_category")["summer_cap_mw"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "wb_mw", "count": "wb_units"})
    .sort_values("wb_mw", ascending=False)
)

# -- PUDL capacity totals --
# Vectorized fuel classification (avoids .apply on PyArrow backend)
fuel_col = gens_operating["fuel_type_code_pudl"].astype(str).str.lower().str.strip()
pm_col = gens_operating["prime_mover_code"].astype(str).str.upper().str.strip()

cc_prime_movers = {"CA", "CS", "CT"}

conditions = [
    fuel_col == "nuclear",
    fuel_col == "coal",
    (fuel_col == "gas") & (pm_col.isin(cc_prime_movers)),
    fuel_col == "gas",
    fuel_col == "oil",
    fuel_col == "solar",
    fuel_col == "wind",
    fuel_col == "hydro",
    fuel_col == "waste",
]
choices = ["Nuclear", "Coal", "Gas CC", "Gas CT/ST", "Oil", "Solar", "Wind", "Hydro", "Biomass"]
gens_operating["fuel_cat_wb"] = np.select(conditions, choices, default="Other")

fuel_col_hr = gens_hr_operating["fuel_type_code_pudl"].astype(str).str.lower().str.strip()
pm_col_hr = gens_hr_operating["prime_mover_code"].astype(str).str.upper().str.strip()
hr_conditions = [
    fuel_col_hr == "nuclear",
    fuel_col_hr == "coal",
    (fuel_col_hr == "gas") & (pm_col_hr.isin(cc_prime_movers)),
    fuel_col_hr == "gas",
    fuel_col_hr == "oil",
    fuel_col_hr == "solar",
    fuel_col_hr == "wind",
    fuel_col_hr == "hydro",
    fuel_col_hr == "waste",
]
gens_hr_operating["fuel_cat_wb"] = np.select(hr_conditions, choices, default="Other")

pudl_cap = (
    gens_operating.groupby("fuel_cat_wb")["summer_capacity_mw"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "pudl_mw", "count": "pudl_units"})
)

# -- Merge and compare --
cap_compare = wb_cap.join(pudl_cap, how="outer").fillna(0)
cap_compare["diff_mw"] = cap_compare["wb_mw"] - cap_compare["pudl_mw"]
cap_compare["diff_pct"] = (cap_compare["diff_mw"] / cap_compare["pudl_mw"].replace(0, np.nan) * 100).round(1)
cap_compare = cap_compare.sort_values("wb_mw", ascending=False)

print("Capacity Comparison: Workbook vs PUDL (EIA 860)")
cap_compare

Capacity Comparison: Workbook vs PUDL (EIA 860)


,wb_mw,wb_units,pudl_mw,pudl_units,diff_mw,diff_pct
fuel_category,,,,,,
Gas CC,59511.791597,313,37498.199219,229,22013.592378,58.7
Coal,46239.570000,142,27314.5,62,18925.07,69.3
Gas CT/ST,36308.355764,684,31734.5,491,4573.855764,14.4
Nuclear,32672.106178,31,33489.601562,32,-817.495384,-2.4
Solar,15522.463311,1644,463.100006,166,15059.363305,3251.9
Wind,11122.097392,146,7002.700195,79,4119.397197,58.8
Hydro,8374.772050,308,8367.400391,300,7.371659,0.1
Oil,4919.756129,576,4357.399902,399,562.356227,12.9
Biomass,2003.550000,652,1640.800049,470,362.749951,22.1


In [128]:
# Visualize capacity comparison
cap_plot = cap_compare.reset_index().rename(columns={"index": "fuel_category"})
cap_plot = cap_plot[cap_plot["fuel_category"] != "Storage"]  # PUDL may not have storage mapped

fig = go.Figure()
fig.add_trace(go.Bar(
    x=cap_plot["fuel_category"], y=cap_plot["wb_mw"],
    name="Workbook", marker_color="steelblue",
))
fig.add_trace(go.Bar(
    x=cap_plot["fuel_category"], y=cap_plot["pudl_mw"],
    name="PUDL (EIA 860)", marker_color="coral",
))
fig.update_layout(
    title="Installed Capacity by Fuel Type: Workbook vs PUDL",
    xaxis_title="Fuel Category",
    yaxis_title="Summer Capacity (MW)",
    barmode="group",
    height=500,
)
fig.show()

# Record result (handle NA in diff_pct)
valid_diffs = cap_compare["diff_pct"].dropna()
if len(valid_diffs) == 0:
    record_result("Capacity Totals by Fuel", "WARN", "No comparable fuel categories found")
else:
    max_pct_diff = float(valid_diffs.abs().max())
    if max_pct_diff < 10:
        record_result("Capacity Totals by Fuel", "PASS",
                      f"Max difference: {max_pct_diff:.1f}% (all within 10%)")
    elif max_pct_diff < 25:
        record_result("Capacity Totals by Fuel", "WARN",
                      f"Max difference: {max_pct_diff:.1f}% (some categories >10%)")
    else:
        record_result("Capacity Totals by Fuel", "FAIL",
                      f"Max difference: {max_pct_diff:.1f}% (significant divergence)")

❌ [FAIL] Capacity Totals by Fuel: Max difference: 3251.9% (significant divergence)


In [129]:
# Build a PUDL lookup: plant_name -> total summer capacity
# Aggregate at plant level since workbook units may not have generator-level IDs
pudl_plant_cap = (
    gens_operating
    .groupby(["plant_id_eia", "plant_name_eia"])
    .agg(
        pudl_summer_mw=("summer_capacity_mw", "sum"),
        pudl_units=("generator_id", "count"),
        pudl_fuel=("fuel_type_code_pudl", "first"),
        state=("state", "first"),
    )
    .reset_index()
)

# Normalize names for matching
pudl_plant_cap["name_clean"] = (
    pudl_plant_cap["plant_name_eia"].astype(str).str.lower()
    .str.replace(r"[^a-z0-9\s]", "", regex=True)
    .str.strip()
)

# Workbook: aggregate by plant name
wb_plant_cap = (
    wb.groupby("plant_name")
    .agg(
        wb_summer_mw=("summer_cap_mw", "sum"),
        wb_units=("summer_cap_mw", "count"),
        wb_fuel=("fuel_category", "first"),
        wb_hub=("power_hub", "first"),
    )
    .reset_index()
)
wb_plant_cap["name_clean"] = (
    wb_plant_cap["plant_name"].astype(str).str.lower()
    .str.replace(r"[^a-z0-9\s]", "", regex=True)
    .str.strip()
)

print(f"Workbook plants: {len(wb_plant_cap):,}")
print(f"PUDL plants: {len(pudl_plant_cap):,}")
print(f"Top 20 workbook plants by capacity:")
wb_plant_cap.sort_values("wb_summer_mw", ascending=False).head(20)

Workbook plants: 2,201
PUDL plants: 706
Top 20 workbook plants by capacity:


,plant_name,wb_summer_mw,wb_units,wb_fuel,wb_hub,name_clean
157,"Bath County (Bath, VA)",3003.000000,12,Hydro,PJM Dominion,bath county bath va
1160,"John E Amos (Putnam, WV)",2900.000000,3,Coal,PJM AEP (Dayton),john e amos putnam wv
942,"Gavin Power, LLC (Gallia, OH)",2691.000000,2,Coal,PJM AEP (Dayton),gavin power llc gallia oh
1738,"Rockport (Spencer, IN)",2600.000000,2,Coal,PJM AEP (Dayton),rockport spencer in
1599,"Peach Bottom (York, PA)",2549.400000,2,Nuclear,PJM METED,peach bottom york pa
1980,"TalenEnergy Susquehanna (Luzerne, PA)",2494.000000,2,Nuclear,PJM PPL,talenenergy susquehanna luzerne pa
226,"Braidwood Generation Station (Will, IL)",2337.000000,2,Nuclear,PJM COMED,braidwood generation station will il
1575,"PSEG Salem Generating Station (Salem, NJ)",2323.097021,3,Oil,PJM AE,pseg salem generating station salem nj
324,"Chalk Point Power (Prince Georges, MD)",2301.000000,10,Gas CT/ST,PJM Dominion,chalk point power prince georges md
265,"Byron Generating Station (Ogle, IL)",2300.000000,2,Nuclear,PJM COMED,byron generating station ogle il


In [130]:
# Attempt exact name match
matched = wb_plant_cap.merge(
    pudl_plant_cap, on="name_clean", how="inner", suffixes=("_wb", "_pudl")
)
matched["cap_diff_mw"] = matched["wb_summer_mw"] - matched["pudl_summer_mw"]
matched["cap_diff_pct"] = (
    matched["cap_diff_mw"] / matched["pudl_summer_mw"].replace(0, np.nan) * 100
).round(1)

n_matched = len(matched)
n_wb_total = len(wb_plant_cap)
match_rate = n_matched / n_wb_total * 100

print(f"Exact name matches: {n_matched:,} / {n_wb_total:,} ({match_rate:.1f}%)")
print(f"Matched capacity: {matched['wb_summer_mw'].sum():,.0f} MW")
print(f"Total workbook capacity: {wb_plant_cap['wb_summer_mw'].sum():,.0f} MW")

# Show largest discrepancies
big_diffs = matched.loc[
    matched["cap_diff_pct"].abs() > 10,
    ["plant_name", "wb_fuel", "wb_hub", "wb_summer_mw", "pudl_summer_mw",
     "cap_diff_mw", "cap_diff_pct"]
].sort_values("cap_diff_mw", key=abs, ascending=False)

print(f"\nPlants with >10% capacity difference ({len(big_diffs)}):")
big_diffs.head(20)

Exact name matches: 226 / 2,201 (10.3%)
Matched capacity: 12,430 MW
Total workbook capacity: 217,474 MW

Plants with >10% capacity difference (9):


,plant_name,wb_fuel,wb_hub,wb_summer_mw,pudl_summer_mw,cap_diff_mw,cap_diff_pct
72,Haverhill North Cogeneration Facility,Gas CT/ST,PJM AEP (Dayton),53.0,34.0,19.0,55.9
176,Shawnee (PA),Oil,PJM PPL,17.0,20.6,-3.6,-17.5
36,Cinnamon Bay Edgeboro Landfill,Biomass,PJM PSEG,9.1,6.0,3.1,51.7
70,Hamilton (PA),Oil,PJM METED,18.0,20.4,-2.4,-11.8
146,Orrtanna,Oil,PJM METED,18.0,20.200001,-2.200001,-10.9
219,William Paterson University,Solar,PJM JCPL,1.4,2.3,-0.9,-39.1
196,Titusville Solar,Solar,PJM PSEG,4.1,3.5,0.6,17.1
160,Pocono Solar Project,Solar,PJM PPL,3.0,2.5,0.5,20.0
59,FedEx Woodbridge,Solar,PJM PSEG,2.4,2.0,0.4,20.0


In [131]:
# Unmatched workbook plants (not in PUDL by name)
unmatched_wb = wb_plant_cap[
    ~wb_plant_cap["name_clean"].isin(matched["name_clean"])
].sort_values("wb_summer_mw", ascending=False)

print(f"Unmatched workbook plants: {len(unmatched_wb):,} ({unmatched_wb['wb_summer_mw'].sum():,.0f} MW)")
print("\nTop 20 unmatched (largest capacity):")
unmatched_wb.head(20)

# NOTE: Many unmatched plants may just have different naming conventions.
# A more robust approach would use EIA plant IDs, but the workbook uses names only.

Unmatched workbook plants: 1,975 (205,044 MW)

Top 20 unmatched (largest capacity):


,plant_name,wb_summer_mw,wb_units,wb_fuel,wb_hub,name_clean
157,"Bath County (Bath, VA)",3003.000000,12,Hydro,PJM Dominion,bath county bath va
1160,"John E Amos (Putnam, WV)",2900.000000,3,Coal,PJM AEP (Dayton),john e amos putnam wv
942,"Gavin Power, LLC (Gallia, OH)",2691.000000,2,Coal,PJM AEP (Dayton),gavin power llc gallia oh
1738,"Rockport (Spencer, IN)",2600.000000,2,Coal,PJM AEP (Dayton),rockport spencer in
1599,"Peach Bottom (York, PA)",2549.400000,2,Nuclear,PJM METED,peach bottom york pa
1980,"TalenEnergy Susquehanna (Luzerne, PA)",2494.000000,2,Nuclear,PJM PPL,talenenergy susquehanna luzerne pa
226,"Braidwood Generation Station (Will, IL)",2337.000000,2,Nuclear,PJM COMED,braidwood generation station will il
1575,"PSEG Salem Generating Station (Salem, NJ)",2323.097021,3,Oil,PJM AE,pseg salem generating station salem nj
324,"Chalk Point Power (Prince Georges, MD)",2301.000000,10,Gas CT/ST,PJM Dominion,chalk point power prince georges md
265,"Byron Generating Station (Ogle, IL)",2300.000000,2,Nuclear,PJM COMED,byron generating station ogle il


In [132]:
# Scatter plot: matched plant capacities
if len(matched) > 0:
    fig = px.scatter(
        matched, x="pudl_summer_mw", y="wb_summer_mw",
        color="wb_fuel",
        color_discrete_map=FUEL_COLORS,
        hover_data=["plant_name", "wb_hub", "cap_diff_pct"],
        title="Plant Capacity: Workbook vs PUDL (matched plants)",
        labels={"pudl_summer_mw": "PUDL Summer Cap (MW)", "wb_summer_mw": "Workbook Summer Cap (MW)"},
    )
    # Add 1:1 line
    max_val = max(matched["pudl_summer_mw"].max(), matched["wb_summer_mw"].max()) * 1.05
    fig.add_shape(type="line", x0=0, y0=0, x1=max_val, y1=max_val,
                  line=dict(dash="dash", color="gray"))
    fig.update_layout(height=600, width=800)
    fig.show()

    # Record result
    within_10 = (matched["cap_diff_pct"].abs() <= 10).sum()
    record_result("Plant-Level Capacity Match", "PASS" if match_rate > 50 else "WARN",
                  f"{n_matched} plants matched ({match_rate:.0f}%); "
                  f"{within_10}/{n_matched} within 10% of PUDL")

⚠️ [WARN] Plant-Level Capacity Match: 226 plants matched (10%); 217/226 within 10% of PUDL


---
## 4. Heat Rate Comparison

Compare workbook heat rates (column G, units unclear) against PUDL implied
heat rates (`unit_heat_rate_mmbtu_per_mwh`). Only thermal generators have heat rates.

In [133]:
# Workbook heat rate distribution by fuel type (thermal only)
thermal_fuels = ["Nuclear", "Coal", "Gas CC", "Gas CT/ST", "Oil"]
wb_thermal = wb[wb["fuel_category"].isin(thermal_fuels) & (wb["heat_rate"] > 0)].copy()

# PUDL heat rate distribution (thermal generators with valid HR)
pudl_thermal = gens_hr_operating[
    gens_hr_operating["fuel_cat_wb"].isin(thermal_fuels)
    & (gens_hr_operating["unit_heat_rate_mmbtu_per_mwh"].notna())
    & (gens_hr_operating["unit_heat_rate_mmbtu_per_mwh"] > 0)
].copy()

print(f"Heat rate comparison year: {latest_hr_year}")
print("Workbook Heat Rate Summary (by fuel):")
print(wb_thermal.groupby("fuel_category")["heat_rate"].describe().round(2))
print("\nPUDL Heat Rate Summary (MMBtu/MWh, by fuel):")
print(pudl_thermal.groupby("fuel_cat_wb")["unit_heat_rate_mmbtu_per_mwh"].describe().round(2))

Heat rate comparison year: 2024-01-01 00:00:00
Workbook Heat Rate Summary (by fuel):
               count   mean   std   min    25%    50%    75%    max
fuel_category                                                      
Coal           125.0  11.32  2.81  4.42   9.87  10.39  11.17  24.88
Gas CC         302.0   8.48  2.89  3.36   7.04   7.25   8.49  19.54
Gas CT/ST      531.0  12.10  3.46  0.13  10.46  11.67  13.31  24.73
Nuclear         31.0   9.57  0.04  9.47   9.56   9.57   9.60   9.64
Oil            341.0  13.14  4.59  2.76   9.84  12.50  15.87  25.00

PUDL Heat Rate Summary (MMBtu/MWh, by fuel):
             count   mean   std    min    25%    50%    75%    max
fuel_cat_wb                                                       
Coal          55.0  13.05  3.53   8.99  10.51  11.65  15.05  22.57
Gas CC       221.0   7.86  1.47   6.06    7.1   7.39   8.27  13.91
Gas CT/ST     50.0  12.57   8.5   5.55   8.98  11.37  12.71  54.61
Oil            7.0   14.3   4.8  10.39  10.39  12.26  16.9

In [134]:
# Side-by-side box plots
fig = make_subplots(rows=1, cols=2, subplot_titles=["Workbook Heat Rates", "PUDL Heat Rates (MMBtu/MWh)"])

for fuel in thermal_fuels:
    wb_vals = wb_thermal.loc[wb_thermal["fuel_category"] == fuel, "heat_rate"]
    pudl_vals = pudl_thermal.loc[pudl_thermal["fuel_cat_wb"] == fuel, "unit_heat_rate_mmbtu_per_mwh"]
    color = FUEL_COLORS.get(fuel, "gray")
    
    fig.add_trace(go.Box(y=wb_vals, name=fuel, marker_color=color,
                         showlegend=False), row=1, col=1)
    fig.add_trace(go.Box(y=pudl_vals, name=fuel, marker_color=color,
                         showlegend=True), row=1, col=2)

fig.update_yaxes(title_text="Heat Rate", row=1, col=1, range=[0, 25])
fig.update_yaxes(title_text="Heat Rate (MMBtu/MWh)", row=1, col=2, range=[0, 25])
fig.update_layout(title="Heat Rate Distributions: Workbook vs PUDL", height=550)
fig.show()

# Check if workbook HR values are in MMBtu/MWh range (5-20) or BTU/kWh range (5000-20000)
median_hr = wb_thermal["heat_rate"].median()
if median_hr < 50:
    hr_unit = "MMBtu/MWh"
    record_result("Heat Rate Units", "WARN",
                  f"Median HR = {median_hr:.1f} — values are in MMBtu/MWh, "
                  f"but column header says BTU/kWh. Formula /1000 is wrong.")
else:
    hr_unit = "BTU/kWh"
    record_result("Heat Rate Units", "PASS",
                  f"Median HR = {median_hr:.0f} — consistent with BTU/kWh header.")

⚠️ [WARN] Heat Rate Units: Median HR = 10.7 — values are in MMBtu/MWh, but column header says BTU/kWh. Formula /1000 is wrong.


In [135]:
# Direct HR comparison for matched plants (thermal only)
# Merge workbook units with PUDL on cleaned plant name
wb_hr = wb_thermal[["plant_name", "fuel_category", "summer_cap_mw", "heat_rate"]].copy()
wb_hr["name_clean"] = (
    wb_hr["plant_name"].astype(str).str.lower()
    .str.replace(r"[^a-z0-9\s]", "", regex=True).str.strip()
)

# PUDL: capacity-weighted average HR per plant
pudl_hr = (
    pudl_thermal
    .assign(weighted_hr=lambda x: x["unit_heat_rate_mmbtu_per_mwh"] * x["summer_capacity_mw"])
    .groupby("plant_name_eia")
    .agg(
        pudl_hr_wavg=("weighted_hr", "sum"),
        pudl_cap_total=("summer_capacity_mw", "sum"),
    )
    .reset_index()
)
pudl_hr["pudl_hr"] = pudl_hr["pudl_hr_wavg"] / pudl_hr["pudl_cap_total"].replace(0, np.nan)
pudl_hr["name_clean"] = (
    pudl_hr["plant_name_eia"].astype(str).str.lower()
    .str.replace(r"[^a-z0-9\s]", "", regex=True).str.strip()
)

hr_compare = wb_hr.merge(pudl_hr[["name_clean", "pudl_hr"]], on="name_clean", how="inner")
hr_compare["hr_diff"] = hr_compare["heat_rate"] - hr_compare["pudl_hr"]
hr_compare["hr_diff_pct"] = (hr_compare["hr_diff"] / hr_compare["pudl_hr"].replace(0, np.nan) * 100).round(1)

print(f"Heat rate matches: {len(hr_compare):,} units")
print(f"\nHR difference distribution (workbook - PUDL):")
print(hr_compare["hr_diff_pct"].describe().round(1))

# Flag outliers
hr_outliers = hr_compare[hr_compare["hr_diff_pct"].abs() > 20]
print(f"\nUnits with HR diff >20%: {len(hr_outliers)}")
if len(hr_outliers) > 0:
    hr_outliers.sort_values("hr_diff_pct", key=abs, ascending=False).head(15)

Heat rate matches: 10 units

HR difference distribution (workbook - PUDL):
count    10.0
mean      1.5
std      12.1
min     -18.9
25%      -2.5
50%      -0.8
75%       4.8
max      29.1
Name: hr_diff_pct, dtype: double[pyarrow]

Units with HR diff >20%: 1


---
## 5. Fuel Cost Assumptions vs EIA 923 Delivered Costs

Compare the workbook Assumptions sheet fuel prices against actual
delivered fuel costs reported to EIA by PJM plants.

In [136]:
%%time
fuel_receipts = read_pudl("out_eia923__fuel_receipts_costs")
fuel_pjm = fuel_receipts[fuel_receipts.plant_id_eia.isin(pjm_plant_ids)].copy()

# Convert pyarrow date to pandas datetime for comparison
fuel_pjm["report_date"] = pd.to_datetime(fuel_pjm["report_date"])

# Use the most recent 12 months of data
max_date = fuel_pjm.report_date.max()
cutoff = max_date - pd.DateOffset(months=12)
fuel_recent = fuel_pjm[fuel_pjm.report_date > cutoff].copy()

print(f"Fuel receipts date range: {fuel_pjm.report_date.min()} to {max_date}")
print(f"Using last 12 months: {cutoff.date()} to {max_date.date()}")
print(f"Records: {len(fuel_recent):,}")

Fuel receipts date range: 2008-01-01 00:00:00 to 2025-11-01 00:00:00
Using last 12 months: 2024-11-01 to 2025-11-01
Records: 2,338
CPU times: total: 922 ms
Wall time: 2.08 s


In [137]:
# Aggregate by fuel type
fuel_summary = (
    fuel_recent
    .groupby("fuel_type_code_pudl")["fuel_cost_per_mmbtu"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .round(2)
)
print("EIA 923 Delivered Fuel Costs ($/MMBtu) — PJM, last 12 months:\n")
fuel_summary

EIA 923 Delivered Fuel Costs ($/MMBtu) — PJM, last 12 months:



,count,mean,std,min,10%,25%,50%,75%,90%,max
fuel_type_code_pudl,,,,,,,,,,
coal,685.0,3.05,1.04,0.0,2.2,2.25,2.9,3.43,4.31,7.81
gas,1218.0,87.61,2843.77,0.0,2.3,2.75,3.25,4.12,5.8,99241.67
hydro,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
nuclear,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
oil,234.0,17.69,1.91,14.83,15.79,16.72,17.52,18.41,19.24,38.07
other,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
solar,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
waste,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
wind,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [138]:
# Box plots of delivered fuel costs (only fuel types with receipts data)
thermal_fuel_codes = ["coal", "gas", "oil"]
fuel_plot = fuel_recent[fuel_recent["fuel_type_code_pudl"].isin(thermal_fuel_codes)].copy()
fuel_plot["fuel_cost_per_mmbtu"] = pd.to_numeric(fuel_plot["fuel_cost_per_mmbtu"], errors="coerce")
fuel_plot = fuel_plot.dropna(subset=["fuel_cost_per_mmbtu"])

# Only plot fuel types that actually have data
available_fuels = fuel_plot["fuel_type_code_pudl"].unique().tolist()

fig = go.Figure()
fuel_colors_eia = {"coal": "#555555", "gas": "#d62728", "oil": "#17becf"}
for fuel in available_fuels:
    subset = fuel_plot[fuel_plot["fuel_type_code_pudl"] == fuel]
    fig.add_trace(go.Box(
        y=subset["fuel_cost_per_mmbtu"], name=fuel,
        marker_color=fuel_colors_eia.get(fuel, "gray"),
    ))

fig.update_layout(
    title="EIA 923 Delivered Fuel Costs to PJM Plants (last 12 months)",
    yaxis_title="$/MMBtu",
    height=550, showlegend=False,
)

# Add workbook assumption price annotations
gas_prices = wb.loc[
    (wb["fuel_category"].isin(["Gas CC", "Gas CT/ST"])) & (wb["fuel_price"] > 0),
    "fuel_price"
]
coal_prices = wb.loc[
    (wb["fuel_category"] == "Coal") & (wb["fuel_price"] > 0),
    "fuel_price"
]
oil_prices = wb.loc[
    (wb["fuel_category"] == "Oil") & (wb["fuel_price"] > 0),
    "fuel_price"
]

for fuel_code, prices in [("gas", gas_prices), ("coal", coal_prices), ("oil", oil_prices)]:
    if fuel_code in available_fuels and len(prices) > 0:
        for val in prices.unique():
            fig.add_annotation(
                x=fuel_code, y=float(val),
                text=f"WB: ${float(val):.2f}",
                showarrow=True, arrowhead=2, arrowcolor="red",
                font=dict(size=9, color="red"),
            )

fig.update_yaxes(range=[0, 25])
fig.show()

In [139]:
# Quantitative check: are workbook gas prices within IQR of delivered costs?
gas_eia = pd.to_numeric(
    fuel_recent.loc[fuel_recent["fuel_type_code_pudl"] == "gas", "fuel_cost_per_mmbtu"],
    errors="coerce"
).dropna()

if len(gas_eia) > 0 and len(gas_prices) > 0:
    q25, q75 = float(gas_eia.quantile(0.25)), float(gas_eia.quantile(0.75))
    wb_gas_min, wb_gas_max = float(gas_prices.min()), float(gas_prices.max())
    eia_median = float(gas_eia.median())

    in_range = (wb_gas_min >= q25 * 0.8) and (wb_gas_max <= q75 * 1.2)
    record_result(
        "Gas Price Assumptions",
        "PASS" if in_range else "WARN",
        f"Workbook: ${wb_gas_min:.2f}-${wb_gas_max:.2f}/MMBtu; "
        f"EIA median: ${eia_median:.2f}, IQR: ${q25:.2f}-${q75:.2f}"
    )

coal_eia = pd.to_numeric(
    fuel_recent.loc[fuel_recent["fuel_type_code_pudl"] == "coal", "fuel_cost_per_mmbtu"],
    errors="coerce"
).dropna()

if len(coal_eia) > 0 and len(coal_prices) > 0:
    q25, q75 = float(coal_eia.quantile(0.25)), float(coal_eia.quantile(0.75))
    wb_coal_min, wb_coal_max = float(coal_prices.min()), float(coal_prices.max())
    eia_median = float(coal_eia.median())

    in_range = (wb_coal_min >= q25 * 0.7) and (wb_coal_max <= q75 * 1.3)
    record_result(
        "Coal Price Assumptions",
        "PASS" if in_range else "WARN",
        f"Workbook: ${wb_coal_min:.2f}-${wb_coal_max:.2f}/MMBtu; "
        f"EIA median: ${eia_median:.2f}, IQR: ${q25:.2f}-${q75:.2f}"
    )

✅ [PASS] Gas Price Assumptions: Workbook: $2.45-$2.90/MMBtu; EIA median: $3.25, IQR: $2.75-$4.12
⚠️ [WARN] Coal Price Assumptions: Workbook: $1.20-$2.80/MMBtu; EIA median: $2.90, IQR: $2.25-$3.43


---
## 6. VOM Reasonableness vs FERC Form 1 Nonfuel O&M

PUDL does not provide generator-level VOM. The closest proxy is FERC Form 1
plant-level nonfuel O&M per MWh. This is a reasonableness check, not an exact match.

In [140]:
%%time
# Load FERC Form 1 steam plant data
ferc1 = read_pudl("out_ferc1__yearly_steam_plants_sched402")

# Get PUDL plant IDs for PJM (via EIA mapping)
pudl_id_map = gens_operating[["plant_id_eia", "plant_id_pudl"]].drop_duplicates()
pjm_pudl_ids = pudl_id_map["plant_id_pudl"].dropna().unique()

ferc1_pjm = ferc1[ferc1["plant_id_pudl"].isin(pjm_pudl_ids)].copy()
ferc1_max_year = int(ferc1_pjm["report_year"].max())
ferc1_latest = ferc1_pjm[ferc1_pjm["report_year"] == ferc1_max_year].copy()

print(f"FERC1 PJM steam plants: {len(ferc1_latest):,}")
print(f"Report year: {ferc1_max_year}")

FERC1 PJM steam plants: 51
Report year: 2024
CPU times: total: 46.9 ms
Wall time: 748 ms


In [141]:
# Distribution of nonfuel O&M per MWh
ferc1_vom = pd.to_numeric(ferc1_latest["opex_nonfuel_per_mwh"], errors="coerce").dropna()
ferc1_vom = ferc1_vom[(ferc1_vom > 0) & (ferc1_vom < 100)]  # Remove outliers

# Workbook VOM distribution
wb_vom = wb.loc[wb["vom"] > 0, "vom"].astype(float)

print("FERC1 Nonfuel O&M per MWh (PJM steam plants):")
print(ferc1_vom.describe().round(2))
print(f"\nWorkbook VOM ($/MWh):")
print(wb_vom.describe().round(2))

FERC1 Nonfuel O&M per MWh (PJM steam plants):
count     47.0
mean     16.86
std      18.86
min        0.0
25%       4.38
50%      13.97
75%      18.65
max      97.94
Name: opex_nonfuel_per_mwh, dtype: double[pyarrow]

Workbook VOM ($/MWh):
count    4593.00
mean        3.11
std         2.22
min         1.00
25%         1.05
50%         2.59
75%         4.40
max        10.42
Name: vom, dtype: float64


In [142]:
# Overlay histograms
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=wb_vom.values, name="Workbook VOM", opacity=0.6,
    marker_color="steelblue", nbinsx=40,
))
fig.add_trace(go.Histogram(
    x=ferc1_vom.values, name="FERC1 Nonfuel O&M/MWh", opacity=0.6,
    marker_color="coral", nbinsx=40,
))
fig.update_layout(
    title="VOM Distribution: Workbook vs FERC Form 1 Nonfuel O&M",
    xaxis_title="$/MWh",
    yaxis_title="Count",
    barmode="overlay",
    height=450,
)
fig.update_xaxes(range=[0, 30])
fig.show()

# Record result
if len(wb_vom) > 0 and len(ferc1_vom) > 0:
    wb_median = float(wb_vom.median())
    ferc_median = float(ferc1_vom.median())
    record_result(
        "VOM Reasonableness",
        "PASS" if abs(wb_median - ferc_median) < ferc_median * 0.5 else "WARN",
        f"Workbook VOM median: ${wb_median:.2f}/MWh; "
        f"FERC1 nonfuel O&M median: ${ferc_median:.2f}/MWh. "
        f"Note: FERC is plant-level (fixed+variable), workbook is generator-level VOM only."
    )
else:
    record_result("VOM Reasonableness", "WARN", "Insufficient data for comparison")

⚠️ [WARN] VOM Reasonableness: Workbook VOM median: $2.59/MWh; FERC1 nonfuel O&M median: $13.97/MWh. Note: FERC is plant-level (fixed+variable), workbook is generator-level VOM only.


---
## 7. Fuel Category / Prime Mover Classification

Verify that the workbook's fuel category assignments match PUDL's fuel type
and prime mover codes for matched plants.

In [143]:
# PUDL fuel mix by workbook fuel category
pudl_fuel_mix = (
    gens_operating
    .groupby(["fuel_cat_wb", "fuel_type_code_pudl", "prime_mover_code"])
    .agg(n_gens=("generator_id", "count"), total_mw=("summer_capacity_mw", "sum"))
    .reset_index()
)
# Convert pyarrow types to native Python for formatting
pudl_fuel_mix["total_mw"] = pd.to_numeric(pudl_fuel_mix["total_mw"], errors="coerce")
pudl_fuel_mix["n_gens"] = pd.to_numeric(pudl_fuel_mix["n_gens"], errors="coerce").astype(int)
pudl_fuel_mix = pudl_fuel_mix.sort_values(["fuel_cat_wb", "total_mw"], ascending=[True, False])

print("PUDL PJM Generators by Mapped Fuel Category + Prime Mover:\n")
for cat in thermal_fuels + ["Hydro", "Wind", "Solar", "Biomass"]:
    subset = pudl_fuel_mix[pudl_fuel_mix["fuel_cat_wb"] == cat]
    if len(subset) > 0:
        total = float(subset["total_mw"].sum())
        print(f"\n{cat} ({total:,.0f} MW):")
        for _, r in subset.iterrows():
            pm = str(r["prime_mover_code"])
            ft = str(r["fuel_type_code_pudl"])
            ng = int(r["n_gens"])
            mw = float(r["total_mw"])
            print(f"  {ft:>10s}  {pm:>4s}  {ng:>5,d} gens  {mw:>10,.0f} MW")

PUDL PJM Generators by Mapped Fuel Category + Prime Mover:


Nuclear (33,490 MW):
     nuclear    ST     32 gens      33,490 MW

Coal (27,315 MW):
        coal    ST     60 gens      27,297 MW
        coal    OT      2 gens          18 MW

Gas CC (37,498 MW):
         gas    CT    152 gens      21,926 MW
         gas    CA     76 gens      13,982 MW
         gas    CS      4 gens       1,590 MW

Gas CT/ST (31,734 MW):
         gas    GT    298 gens      22,394 MW
         gas    ST     93 gens       9,104 MW
         gas    IC     93 gens         209 MW
         gas    FC      7 gens          28 MW

Oil (4,357 MW):
         oil    GT    116 gens       2,875 MW
         oil    ST      3 gens         855 MW
         oil    IC    277 gens         473 MW
         oil    CT      2 gens         104 MW
         oil    CA      1 gens          50 MW

Hydro (8,367 MW):
       hydro    PS     22 gens       5,161 MW
       hydro    HY    278 gens       3,207 MW

Wind (7,003 MW):
        wind    WT

In [144]:
# Workbook fuel category breakdown
wb_fuel_mix = (
    wb.groupby(["fuel_category", "unit_type"])
    .agg(n_units=("summer_cap_mw", "count"), total_mw=("summer_cap_mw", "sum"))
    .reset_index()
    .sort_values(["fuel_category", "total_mw"], ascending=[True, False])
)

print("Workbook Fuel Category + Unit Type:\n")
for cat in wb_fuel_mix["fuel_category"].unique():
    subset = wb_fuel_mix[wb_fuel_mix["fuel_category"] == cat]
    total = subset["total_mw"].sum()
    print(f"\n{cat} ({total:,.0f} MW):")
    for _, r in subset.iterrows():
        print(f"  {r['unit_type']:>25s}  {r['n_units']:>5,d} units  {r['total_mw']:>10,.0f} MW")

record_result("Fuel Classification", "PASS",
              "Fuel categories reviewed. Cross-check PUDL prime mover codes above for mismatches.")

Workbook Fuel Category + Unit Type:


Biomass (2,004 MW):
            Biomass / Waste    652 units       2,004 MW

Coal (46,240 MW):
                 Bituminous    133 units      39,795 MW
             Sub-Bituminous      9 units       6,445 MW

Gas CC (59,512 MW):
                     Gas CC    313 units      59,512 MW

Gas CT/ST (36,308 MW):
             Gas OC / Steam    669 units      36,267 MW
                      Other     15 units          42 MW

Hydro (8,375 MW):
                    Storage     28 units       5,183 MW
                      Hydro    280 units       3,192 MW

Nuclear (32,672 MW):
                    Nuclear     31 units      32,672 MW

Oil (4,920 MW):
                        Oil    576 units       4,920 MW

Other (113 MW):
                      Other      5 units         113 MW

Solar (15,522 MW):
                      Solar  1,137 units      11,007 MW
                 Solar (DG)    507 units       4,516 MW

Storage (686 MW):
                    Storage     92 u

---
## 8. Operational Status — Retired or Missing Units

Check if any workbook units appear as retired in PUDL, and whether any
large PJM generators in PUDL are missing from the workbook.

In [145]:
# All PJM generators (including non-operating) in latest year
gens_all_latest = gens_pjm[gens_pjm.report_date == latest_year].copy()

# Recently retired generators (retirement date in last 3 years)
three_yrs_ago = latest_year - pd.DateOffset(years=3)
gens_all_latest["retirement_dt"] = pd.to_datetime(gens_all_latest["generator_retirement_date"], errors="coerce")
retired = gens_all_latest[
    (gens_all_latest["operational_status"] != "existing")
    & (gens_all_latest["retirement_dt"].notna())
    & (gens_all_latest["retirement_dt"] > three_yrs_ago)
].copy()

# Check if any retired plant names appear in the workbook
retired["name_clean"] = (
    retired["plant_name_eia"].astype(str).str.lower()
    .str.replace(r"[^a-z0-9\s]", "", regex=True).str.strip()
)
wb_names = set(
    wb["plant_name"].astype(str).str.lower()
    .str.replace(r"[^a-z0-9\s]", "", regex=True).str.strip()
)

retired_in_wb = retired[retired["name_clean"].isin(wb_names)]

print(f"Recently retired PJM generators (last 3 years): {len(retired):,}")
if len(retired_in_wb) > 0:
    print()
    print(f"WARNING: {len(retired_in_wb)} retired generators found in workbook:")
    print(retired_in_wb[["plant_name_eia", "generator_id", "summer_capacity_mw",
                         "fuel_type_code_pudl", "generator_retirement_date"]].to_string())
    record_result("Retired Units in Workbook", "WARN",
                  f"{len(retired_in_wb)} recently retired generators still in workbook")
else:
    print("No retired generators found in workbook (by name match).")
    record_result("Retired Units in Workbook", "PASS", "No retired units detected.")

Recently retired PJM generators (last 3 years): 177

                                 plant_name_eia generator_id  summer_capacity_mw fuel_type_code_pudl generator_retirement_date
293784                        Mickleton Station         MICK           63.700001                 gas                2024-06-01
322503    AES Warrior Run Cogeneration Facility         GEN1               180.0                coal                2024-06-01
341923             Bunge North America East LLC         3516                 2.2                 gas                2025-07-01
523247                        PSEG Hackettstown            1                 2.0               solar                2024-09-01
533072  Trexlertown Solar Array North and South         GEN1                 1.9               solar                2023-02-01
537021                  Union City Wind Turbine            1                 1.0                wind                2023-03-01
537037     Randolph Eastern School Wind Turbine           

In [146]:
# Large operating PJM generators missing from workbook
pudl_names = set(
    gens_operating["plant_name_eia"].astype(str).str.lower()
    .str.replace(r"[^a-z0-9\s]", "", regex=True).str.strip()
)

missing_from_wb = gens_operating[
    ~gens_operating["plant_name_eia"].astype(str).str.lower()
    .str.replace(r"[^a-z0-9\s]", "", regex=True).str.strip()
    .isin(wb_names)
].copy()

missing_large = (
    missing_from_wb
    .groupby(["plant_name_eia", "plant_id_eia"])
    .agg(total_mw=("summer_capacity_mw", "sum"), fuel=("fuel_type_code_pudl", "first"))
    .reset_index()
    .sort_values("total_mw", ascending=False)
)

# Only flag plants > 100 MW
missing_large_100 = missing_large[missing_large["total_mw"] >= 100]
print(f"Operating PUDL plants not matched in workbook: {len(missing_large):,}")
print(f"Of those, >= 100 MW: {len(missing_large_100):,} ({missing_large_100['total_mw'].sum():,.0f} MW)")
print("\nTop 20 missing large plants:")
missing_large_100.head(20)

# NOTE: Many may just be naming differences. This list is for manual review.

Operating PUDL plants not matched in workbook: 480
Of those, >= 100 MW: 173 (133,569 MW)

Top 20 missing large plants:


,plant_name_eia,plant_id_eia,total_mw,fuel
22,Bath County,6167,3015.100098,hydro
231,John E Amos,3935,2900.0,coal
180,"Gavin Power, LLC",8102,2722.0,coal
387,Rockport,6166,2600.0,coal
346,Peach Bottom,3166,2549.399902,nuclear
424,TalenEnergy Susquehanna,6103,2494.0,nuclear
43,Braidwood Generation Station,6022,2332.300049,nuclear
337,PSEG Salem Generating Station,2410,2323.100098,nuclear
61,Byron Generating Station,6023,2300.0,nuclear
242,LaSalle Generating Station,6026,2264.399902,nuclear


---
## 9. Wind & Solar Installed Capacity

Compare workbook renewable capacity totals against PUDL.

In [147]:
# Workbook renewable totals
wb_wind = wb.loc[wb["fuel_category"] == "Wind", "summer_cap_mw"].sum()
wb_solar = wb.loc[wb["fuel_category"] == "Solar", "summer_cap_mw"].sum()
wb_hydro = wb.loc[wb["fuel_category"] == "Hydro", "summer_cap_mw"].sum()

# PUDL renewable totals
pudl_wind = gens_operating.loc[gens_operating["fuel_cat_wb"] == "Wind", "summer_capacity_mw"].sum()
pudl_solar = gens_operating.loc[gens_operating["fuel_cat_wb"] == "Solar", "summer_capacity_mw"].sum()
pudl_hydro = gens_operating.loc[gens_operating["fuel_cat_wb"] == "Hydro", "summer_capacity_mw"].sum()

renew_compare = pd.DataFrame({
    "Resource": ["Wind", "Solar", "Hydro"],
    "Workbook MW": [wb_wind, wb_solar, wb_hydro],
    "PUDL MW": [float(pudl_wind), float(pudl_solar), float(pudl_hydro)],
})
renew_compare["Diff MW"] = renew_compare["Workbook MW"] - renew_compare["PUDL MW"]
renew_compare["Diff %"] = (renew_compare["Diff MW"] / renew_compare["PUDL MW"] * 100).round(1)

print("Renewable Capacity Comparison:\n")
renew_compare

Renewable Capacity Comparison:



,Resource,Workbook MW,PUDL MW,Diff MW,Diff %
0,Wind,11122.097392,7002.699986,4119.397407,58.8
1,Solar,15522.463311,463.100000,15059.363311,3251.9
2,Hydro,8374.772050,8367.399963,7.372087,0.1


In [148]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=renew_compare["Resource"], y=renew_compare["Workbook MW"],
    name="Workbook", marker_color="steelblue",
))
fig.add_trace(go.Bar(
    x=renew_compare["Resource"], y=renew_compare["PUDL MW"],
    name="PUDL (EIA 860)", marker_color="coral",
))
fig.update_layout(
    title="Renewable Installed Capacity: Workbook vs PUDL",
    yaxis_title="Summer Capacity (MW)",
    barmode="group", height=450,
)
fig.show()

valid_renew = renew_compare["Diff %"].dropna()
max_diff = float(valid_renew.abs().max()) if len(valid_renew) > 0 else 0.0
record_result("Renewable Capacity Totals", "PASS" if max_diff < 15 else "WARN",
              f"Wind diff: {renew_compare.iloc[0]['Diff %']}%, "
              f"Solar diff: {renew_compare.iloc[1]['Diff %']}%, "
              f"Hydro diff: {renew_compare.iloc[2]['Diff %']}%")

⚠️ [WARN] Renewable Capacity Totals: Wind diff: 58.8%, Solar diff: 3251.9%, Hydro diff: 0.1%


---
## 10. Pass / Fail Summary

In [149]:
results_df = pd.DataFrame(validation_results)

n_pass = (results_df["status"] == "PASS").sum()
n_warn = (results_df["status"] == "WARN").sum()
n_fail = (results_df["status"] == "FAIL").sum()

print("=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)
print(f"  PASS: {n_pass}")
print(f"  WARN: {n_warn}")
print(f"  FAIL: {n_fail}")
print("=" * 80)
print()

results_df

VALIDATION SUMMARY
  PASS: 2
  WARN: 6
  FAIL: 1



,check,status,detail
0,Capacity Totals by Fuel,FAIL,Max difference: 3251.9% (significant divergence)
1,Plant-Level Capacity Match,WARN,226 plants matched (10%); 217/226 within 10% of PUDL
2,Heat Rate Units,WARN,"Median HR = 10.7 — values are in MMBtu/MWh, but column header says BTU/kWh. ..."
3,Gas Price Assumptions,PASS,"Workbook: $2.45-$2.90/MMBtu; EIA median: $3.25, IQR: $2.75-$4.12"
4,Coal Price Assumptions,WARN,"Workbook: $1.20-$2.80/MMBtu; EIA median: $2.90, IQR: $2.25-$3.43"
5,VOM Reasonableness,WARN,Workbook VOM median: $2.59/MWh; FERC1 nonfuel O&M median: $13.97/MWh. Note: ...
6,Fuel Classification,PASS,Fuel categories reviewed. Cross-check PUDL prime mover codes above for misma...
7,Retired Units in Workbook,WARN,9 recently retired generators still in workbook
8,Renewable Capacity Totals,WARN,"Wind diff: 58.8%, Solar diff: 3251.9%, Hydro diff: 0.1%"


In [150]:
# Summary table with color coding
def style_status(val):
    colors = {"PASS": "background-color: #d4edda",
              "WARN": "background-color: #fff3cd",
              "FAIL": "background-color: #f8d7da"}
    return colors.get(val, "")

results_df.style.map(style_status, subset=["status"])

,check,status,detail
0,Capacity Totals by Fuel,FAIL,Max difference: 3251.9% (significant divergence)
1,Plant-Level Capacity Match,WARN,226 plants matched (10%); 217/226 within 10% of PUDL
2,Heat Rate Units,WARN,"Median HR = 10.7 — values are in MMBtu/MWh, but column header says BTU/kWh. Formula /1000 is wrong."
3,Gas Price Assumptions,PASS,"Workbook: $2.45-$2.90/MMBtu; EIA median: $3.25, IQR: $2.75-$4.12"
4,Coal Price Assumptions,WARN,"Workbook: $1.20-$2.80/MMBtu; EIA median: $2.90, IQR: $2.25-$3.43"
5,VOM Reasonableness,WARN,"Workbook VOM median: $2.59/MWh; FERC1 nonfuel O&M median: $13.97/MWh. Note: FERC is plant-level (fixed+variable), workbook is generator-level VOM only."
6,Fuel Classification,PASS,Fuel categories reviewed. Cross-check PUDL prime mover codes above for mismatches.
7,Retired Units in Workbook,WARN,9 recently retired generators still in workbook
8,Renewable Capacity Totals,WARN,"Wind diff: 58.8%, Solar diff: 3251.9%, Hydro diff: 0.1%"


---
## Notes & Caveats

1. **Name matching is imperfect.** The workbook uses plant names without EIA plant IDs.
   Many unmatched plants are likely naming differences, not missing data. Adding `plant_id_eia`
   to the workbook's PJM Raw Data sheet would dramatically improve match rates.

2. **PUDL reporting lag.** EIA 860/923 data has a ~12-18 month lag. Very recently added
   generators may not yet appear in PUDL.

3. **Fuel cost comparison is approximate.** The workbook uses gas hub prices (forward market),
   while EIA 923 reports delivered fuel costs (historical). A $0.30-0.50/MMBtu difference
   is expected due to basis, transport, and timing.

4. **VOM is not directly comparable.** FERC Form 1 `opex_nonfuel_per_mwh` includes both
   fixed and variable O&M allocated across all MWh. Workbook VOM is a generator-level
   variable cost only. NREL ATB provides better technology-level VOM benchmarks.

5. **PUDL does not cover:** PJM zone assignments, must-run status, gas hub pricing,
   planned outages, or energy storage dispatch. These require PJM-specific data sources.

6. **Fuel cost formula bug.** The workbook's Stack Model column J divides by 1000,
   producing fuel costs ~1000x too low. See `.markdowns/PUDL.md` for details and fix.